# Grid Search Hyperparameter Tuning
---
<hr>
**Using GridSearchCV to tune RandomForest on synthetic classification data**

In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import GridSearchCV, train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
print ('imports done\n')

imports done


In [2]:
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
X, y = make_classification(
    n_samples=1000, n_features=20, n_informative=15,
    n_redundant=3, n_classes=2, random_state=42
)
print ('Data shape: %s' % str(X.shape))
print ('Class balance: %.2f / %.2f' % (y.mean(), 1 - y.mean()))

Data shape: (1000, 20)
Class balance: 0.50 / 0.50


In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print ('Train: %d, Test: %d' % (len(X_train), len(X_test)))

Train: 800, Test: 200


<hr>
## **Define Parameter Grid**

In [5]:
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, 15, None]
}
print ('Parameter grid: %s' % str(param_grid))
print ('Total combinations: %d' % (3 * 4))

Parameter grid: {'n_estimators': [50, 100, 200], 'max_depth': [5, 10, 15, None]}
Total combinations: 12


In [6]:
rf = RandomForestClassifier(random_state=42)
grid_search = GridSearchCV(
    rf, param_grid, cv=5, scoring='accuracy',
    n_jobs=-1, verbose=1
)
print ('GridSearchCV initialized\n')

GridSearchCV initialized


<hr>
## **Fit Grid Search**

In [7]:
grid_search.fit(X_train, y_train)
print ('Grid search completed.\n')

Fitting 5 folds for each of 12 candidates, totalling 60 fits
Grid search completed.


In [8]:
print ('Best parameters: %s' % str(grid_search.best_params_))
print ('Best cross-val accuracy: %.4f' % grid_search.best_score_)

Best parameters: {'max_depth': 15, 'n_estimators': 200}
Best cross-val accuracy: 0.9325


<hr>
## **Grid Search Results**

In [9]:
results_df = pd.DataFrame(grid_search.cv_results_)
cols = ['param_n_estimators', 'param_max_depth', 'mean_test_score', 'std_test_score', 'rank_test_score']
print (results_df[cols].to_string(index=False))

param_n_estimators param_max_depth  mean_test_score  std_test_score  rank_test_score
               50              5          0.8875          0.0234               12
               50             10          0.9125          0.0189                8
               50             15          0.9200          0.0156                5
               50           None          0.9187          0.0167                6
              100              5          0.8912          0.0221               11
              100             10          0.9187          0.0178                6
              100             15          0.9263          0.0145                2
              100           None          0.9238          0.0154                4
              200              5          0.8938          0.0212               10
              200             10          0.9212          0.0167                3
              200             15          0.9325          0.0134                1
             

In [10]:
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
test_acc = accuracy_score(y_test, y_pred)
print ('Test accuracy with best model: %.4f' % test_acc)
print ('Confusion matrix:')
print (confusion_matrix(y_test, y_pred))

Test accuracy with best model: 0.9400
Confusion matrix:
[[94  6]
 [ 6 94]]


In [11]:
impo = best_model.feature_importances_
top_idx = np.argsort(impo)[-5:][::-1]
print ('Top 5 feature importances:')
for i in top_idx:
    print ('  Feature %2d: %.4f' % (i, impo[i]))

Top 5 feature importances:
  Feature 12: 0.1245
  Feature  5: 0.1123
  Feature  8: 0.0987
  Feature 15: 0.0891
  Feature  3: 0.0789


<hr>
## **Summary**
**GridSearchCV** successfully tuned RandomForest hyperparameters.
Best config: **n_estimators=200, max_depth=15** with **93.25%** cross-val accuracy.
Test accuracy confirmed at **94.00%**.

In [12]:
print ('Grid search experiment complete.\n')

Grid search experiment complete.
